# Scrap notebooks

This notebook is just a compilation of scrapy tests to view functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sympy as sym

In [ ]:
%matplotlib widget

In [ ]:
phi = 0.3
phi_min = 0.1
phi_max = 0.9

phi_var = sym.symbols('varphi', positive=True) # phi is lower than 1 but no assumption system

alpha = 2
beta = 1
gamma = 4
delta = beta * gamma / (beta + gamma)

alpha_var, beta_var, gamma_var = sym.symbols('alpha beta gamma', positive=True)
delta_expr = beta_var * gamma_var / (beta_var + gamma_var)
delta_var = sym.symbols('delta', positive=True) # delta_var and delta_expr cannot be linked

f = 0.5
f_min = 0
f_max = 10

f_var = sym.symbols('f', nonnegative=True)

E = 0.3
E_min = 0
E_max = 10

E_var = sym.symbols('E', nonnegative=True)

N = 10000
N_min = 1
N_max = 100000

N_var = sym.symbols('N', positive=True)

s = 5000
s_min = 2000
s_max = 10000

s_var = sym.symbols('s', positive=True)

M = 3
M_min = 1
M_max = 8

M_var = sym.symbols('M', positive=True)

Delta_d = 1
Delta_d_min = -5
Delta_d_max = 15

Delta_d_var = sym.symbols('Delta_d', real=True)

Delta_p = 3.2
Delta_p_min = -5
Delta_p_max = 20

Delta_p_var = sym.symbols('Delta_p', real=True)

In [ ]:
dim = 200

# first depending variables
N_ds = np.linspace(0, N, dim)
N_ps = np.linspace(0, N, dim)

N_d_var = sym.symbols('N_d', nonnegative=True)
N_p_var = sym.symbols('N_p', nonnegative=True)
N_s_expr = N_var - N_d_var - N_p_var
N_s_var = sym.symbols('N_s', nonnegative=True) # N_s_var and N_s_expr cannot be linked

N_ds, N_ps = np.meshgrid(N_ds, N_ps)
N_ss = N - N_ds - N_ps

mask = (N_ds >= 0) & (N_ps >= 0) & (N_ss >= 0) & (N_ps <= M * N_ds)

surface = np.zeros((dim, dim), dtype="float")

surface[~mask] = np.nan

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(N_ds, N_ps, surface, cmap=plt.cm.viridis, linewidth=0.2)

ax.set_xlabel('N_d')
ax.set_ylabel('N_p')


In [ ]:
SC = (delta / s) * ((N_ss)**2/(1-phi) + (N_ds**2 + N_ds * N_ps)/phi)\
    + (N_ss + N_ds) * (f + E) + N_ds * Delta_d + N_ps * Delta_p

SC_expr = (delta_var / s_var) * ((N_s_expr)**2/(1-phi_var) + (N_d_var**2 + N_d_var * N_p_var)/phi_var)\
    + (N_s_expr + N_d_var) * (f_var + E_var) + N_d_var * Delta_d_var + N_p_var * Delta_p_var

In [ ]:
SC_expr

In [ ]:
fig = plt.figure()
# some line of codes to see the saddle point

N_ds = np.linspace(1821, 1822, dim)
N_ps = np.linspace(803, 804, dim)
N_ds, N_ps = np.meshgrid(N_ds, N_ps)
N_ss = N - N_ds - N_ps
SC = (delta / s) * ((N_ss)**2/(1-phi) + (N_ds**2 + N_ds * N_ps)/phi)\
   + (N_ss + N_ds) * (f + E) + N_ds * Delta_d + N_ps * Delta_p

SC[(N_ds > 1821.3) & (N_ps > 1821.3- N_ds + 803.8)] = np.nan
SC[(N_ds <1821.7) & (N_ps < 1821.7- N_ds + 803)] = np.nan

ax = fig.add_subplot(projection='3d')
ax.plot_surface(N_ds, N_ps, SC, cmap=plt.cm.viridis, linewidth=0.2)
ax.set_xlabel('N_d')
ax.set_ylabel('N_p')
ax.view_init(20, 0, 0)

In [ ]:
grad_SC = sym.Matrix([SC_expr.diff(var) for var in (N_d_var, N_p_var)])
grad_SC

In [ ]:
hessian_SC = sym.Matrix([[SC_expr.diff(var1).diff(var2) for var2 in (N_d_var, N_p_var)] for var1 in (N_d_var, N_p_var)])
hessian_SC

In [ ]:
hessian_SC_eigen = hessian_SC.eigenvals()
list(hessian_SC_eigen.keys())[0] # which is negative

In [ ]:
list(hessian_SC_eigen.keys())[1] # which is positive

In [ ]:
N_ps_1dim = N_ps[:, 0]
lhs = (f + E + delta * N / s + Delta_d * (1 + phi) / 2) * np.ones(dim)
rhs = delta * N_ps_1dim / ((1-phi) * s)  * (2 - 1/2 * (1+phi)*(1+phi) / phi) + Delta_p

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5, 5), sharex=True, tight_layout=True)
ax1.plot(N_ps_1dim, lhs, label="lhs")
ax1.plot(N_ps_1dim, rhs, label="rhs")

N_ds_1dim_of_N_ps = - N_ps_1dim / 2 * (1+phi) + phi*(1-phi) * (N / (1-phi) - Delta_d * s / (2 * delta))
ax2.plot(N_ps_1dim, N_ds_1dim_of_N_ps, label="N_d")

ax2.set_xlabel('N_p')
ax2.set_ylabel('N_d')

In [ ]:
hessian_SC_eigenvects = hessian_SC.eigenvects()
eigen_values = [hessian_SC_eigenvects[k][0] for k in range(len(hessian_SC_eigenvects)) for l in range (hessian_SC_eigenvects[k][1])]
eigen_vects = [hessian_SC_eigenvects[k][2] for k in range(len(hessian_SC_eigenvects))]

def list_of_vector_to_matrix(list_of_vect):
    matrix = list_of_vect[0]
    for k in range(1, len(list_of_vect)):
        matrix = matrix.row_join(list_of_vect[k])

    return matrix

eigen_vects = [list_of_vector_to_matrix(vect) for vect in eigen_vects]
eigen_vects = list_of_vector_to_matrix(eigen_vects)

diag_matrix = sym.Matrix.diag(*[eigen_values])
# eigen_vects_check, diag_check = hessian_SC.diagonalize()


#### just a few commands to understand basic linear algebra

In [ ]:
# diag_matrix

In [ ]:
# eigen_vects

In [ ]:
# (eigen_vects.inv() * hessian_SC * eigen_vects).applyfunc(sym.cancel)

In [ ]:
# diag_matrix.applyfunc(sym.cancel)

In [ ]:
# eigen_vects_check, diag_check = hessian_SC.diagonalize()

In [ ]:
# (eigen_vects_check.inv() * hessian_SC * eigen_vects_check).applyfunc(sym.cancel)

Ok, j’ai tout compris

Maintenant, on va trouver les solutions.

## Solving

Writing Lagrangian

In [ ]:
# lamb = sym.symbols("lambda")
mu_s, mu_d, mu_p, mu_v = sym.symbols("mu_s mu_d mu_p mu_v", positive=True)

Lag_expr = SC_expr \
    - mu_s * N_s_expr \
    - mu_d * N_d_var \
    - mu_p * N_p_var \
    - mu_v * (M_var * N_d_var - N_p_var)
#     - lamb * (N_var - N_d_var - N_p_var - N_s_expr) \

In [ ]:
Lag_expr

In [ ]:
conditions = [Lag_expr.diff(var) for var in (N_d_var, N_p_var, mu_s, mu_d, mu_p, mu_v)]

In [ ]:
conditions[0]

In [ ]:
conditions[1]

In [ ]:
conditions[2]

In [ ]:
conditions[3]

In [ ]:
conditions[4]

In [ ]:
conditions[5]

In [ ]:
SC_phi_1_expr = (delta_var / s_var) * ((N_d_var**2 + N_d_var * (N_var - N_d_var)))\
    + (N_d_var) * (f_var + E_var) + N_d_var * Delta_d_var + (N_var - N_d_var) * Delta_p_var

In [ ]:
SC_phi_1_expr = SC_phi_1_expr.expand()

In [ ]:
SC_phi_1_expr

In [ ]:
SC_phi_1_expr_diff = SC_phi_1_expr.diff(N_d_var)
SC_phi_1_expr_diff

* Si c’est positif --> on va au min de $\overline{\Delta_d}$, c’est-à-dire $N_d = \frac{N}{M+1}$.
* Si c’est négatif --> On va au max, c’est-à-dire $N_d = N$

### Maintenant on résoud pour $\varphi \in ]0, 1[$